# Run benchmarks
This notebook can be used to run evaluation on MegaDepth-1500 or ScanNet-1500 using different keypoints and clusterings (**note: you need to run clustering with `run_clustering.py` prior to running this script.**).

In [ ]:
DATASET = 'megadepth1500'
KEYPOINTS = 'roma'
CLUSTERING = 'kmeans4d'
NUM_CLUSTERS = 128

THRESHOLDS_PIXELS = {
    'scannet1500': 2.5,
    'megadepth1500': 1.0,
}

SUPERSEED = 0
NUMBER_OF_TRIALS = 10

### Define which methods to evaluate

The function `estimate_relative_pose` has arguments `sampling`, `scoring`, and `refinement`, which accept the values `dense`, `center`, or`approx`. Our main method is `center-center-approx`. Our fastest method is `center-center-center`. The fully dense baseline is `dense-dense-dense`. For details, see the ablation study in the paper.

In [ ]:
from dms.estimation import estimate_relative_pose

METHODS = {
    'dense_baseline': lambda data, s: estimate_relative_pose(data, sampling='dense', scoring='dense', refinement='dense', seed=s),
    'ours_CCC': lambda data, s: estimate_relative_pose(data, 'center', 'center', 'center', seed=s),
    'ours_CCA': lambda data, s: estimate_relative_pose(data, 'center', 'center', 'approx', seed=s),
}

In [ ]:
import os
from tqdm.notebook import tqdm
import numpy as np

from dms.clusters import ClusterCorrespondence
from dms.dataloader import EvaluationDataset
from dms.io import write_h5
from dms.evaluation.runners import run_methods_on_image_pair

In [ ]:
results_path = f'output/{DATASET}_{KEYPOINTS}_{CLUSTERING}{NUM_CLUSTERS}_robust.h5'


In [ ]:
def make_data_dict(image_pair, threshold_pixels):
    (x1, x2) = image_pair.matches()
    (x1c, x2c) = image_pair.matches(calibrated=True)

    K1, K2 = image_pair.calib_matrices()
    cam1, cam2 = image_pair.cameras()

    clusters = image_pair.clusters(min_size=1)
    summarized_matches = [ClusterCorrespondence(c, K1, K2) for c in clusters]

    image_pair_data = {
        'x1': x1,
        'x1c': x1c,
        'x2': x2,
        'x2c': x2c,
        'K1': K1,
        'K2': K2,
        'cam1': cam1,
        'cam2': cam2,
        'f1': np.mean([K1[0,0], K1[1,1]]),
        'f2': np.mean([K2[0,0], K2[1,1]]),
        'clusters': clusters,
        'ecorrs': summarized_matches,
        'key': image_pair.key,
        'image_pair': image_pair,
    }

    if image_pair.has_ground_truth:
        image_pair_data['R_gt'] = image_pair.R()
        image_pair_data['t_gt'] = image_pair.t()

    mean_focal_length = np.mean([image_pair_data['f1'], image_pair_data['f2']])
    opt_params = {
        'threshold_E': threshold_pixels / mean_focal_length,
        'filter_outliers': False,
    }

    return {**image_pair_data, **opt_params}


In [ ]:

if os.path.exists(results_path):
    print(f"Results file {results_path} already exists. Please remove it if you want to re-run becnchmarks.")
    exit(1)
else:
    np.random.seed(SUPERSEED)
    seeds = np.random.randint(0, 1000, NUMBER_OF_TRIALS)

    results = {}
    average_errors = {}
    average_runtimes = {}

    # Create dataset
    data_path = f'data/clustered/{DATASET}_{KEYPOINTS}_{CLUSTERING}{NUM_CLUSTERS}.h5'
    image_dir = f'data/{DATASET}-images/images'
    dataset = EvaluationDataset(data_path, image_dir)

    threshold_pixels = THRESHOLDS_PIXELS[DATASET]

    for image_pair in tqdm(dataset, desc='Running Robust Estimation'):

        # Run solvers
        data = make_data_dict(image_pair, threshold_pixels)
        image_pair_results = run_methods_on_image_pair(METHODS, data, seeds)

        for method_name, method_results in image_pair_results.items():

            # Create dict entries for method (if they do not already exist)
            results.setdefault(method_name, {})
            results[method_name].setdefault('image_pairs', {})
            results[method_name].setdefault('average_runtimes', [])
            results[method_name].setdefault('average_errors', {'R': [], 't': [], 'max': []})
            results[method_name].setdefault('trials', {})
            average_errors.setdefault(method_name, {'R': [], 't': [], 'max': []})
            average_runtimes.setdefault(method_name, [])

            # Log results for image pair
            results[method_name]['image_pairs'][image_pair.key] = method_results

            # Log errors and runtime
            method_errors = method_results['average_errors']
            for k, v in method_errors.items():
                average_errors[method_name][k].append(v)
                results[method_name]['average_errors'][k].append(v)
            average_runtimes[method_name].append(method_results['average_runtime'])
            results[method_name]['average_runtimes'].append(method_results['average_runtime'])

            # Store errors and runtimes per trial
            for trial_name, trial_results in method_results['trials'].items():
                results[method_name]['trials'].setdefault(trial_name, {})

                # Store list of runtimes for trial
                results[method_name]['trials'][trial_name].setdefault('runtimes', []).append(trial_results['runtime'])

                # Store list of errors for trial
                results[method_name]['trials'][trial_name].setdefault('errors', {'R': [], 't': [], 'max': []})
                for k, v in trial_results['errors'].items():
                    results[method_name]['trials'][trial_name]['errors'][k].append(v)

    # Save results
    os.makedirs('output', exist_ok=True)
    write_h5(results, results_path, 'a', verbose=True)

### Print results

In [ ]:
import h5py

from dms.evaluation.reporting import plot_auc_vs_runtime, print_auc_table


def get_errors_and_runtimes_per_trial(results):
    all_errors = {'R': [], 't': [], 'max': []}
    all_runtimes = []

    for trial, trial_results in results['trials'].items():
        errors = trial_results['errors']
        for k, v in errors.items():
            all_errors[k].append(v[()])

        runtimes = trial_results['runtimes'][()]
        all_runtimes.append(runtimes)

    all_errors = {k: np.array(v) for k, v in all_errors.items()}
    return all_errors, np.array(all_runtimes)


def read_results(file_path):
    errors = {}
    runtimes = {}
    average_errors = {}
    average_runtimes = {}

    assert os.path.exists(file_path), f'File not found: {file_path}'
    print('Results from', file_path)

    with h5py.File(file_path, 'r') as f:
        for method_name, method_results in f.items():

            if method_name == 'ground_truth':
                # Skip ground truth data
                continue

            if 'average_errors' in method_results and 'average_runtimes' in method_results:

                # Get errors and runtimes for each trial
                errors[method_name], runtimes[method_name] = get_errors_and_runtimes_per_trial(method_results)

                # Get average errors and runtimes
                average_errors.setdefault(method_name, {})
                for err_name, err in method_results['average_errors'].items():
                    average_errors[method_name][err_name] = err[()]
                average_runtimes[method_name] = method_results['average_runtimes'][()]

            else:
                raise ValueError(f'Could not get errors and runtimes for method {method_name}')

    return errors, runtimes, average_errors, average_runtimes


errors, runtimes, average_errors, average_runtimes = read_results(results_path)
print_auc_table(errors, runtimes, baseline='dense_baseline')
plot_auc_vs_runtime(errors, runtimes)